In [3]:
!pip install torch torchaudio transformers soundfile pandas

V3

In [4]:
import torch
import torch.nn as nn
import torchaudio
import os
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoFeatureExtractor, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, BertTokenizer, Wav2Vec2Processor ,Wav2Vec2FeatureExtractor
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoConfig, Wav2Vec2ForPreTraining
from safetensors.torch import load_file

In [5]:
!cp -r '/kaggle/input/splits-updated-223/splits/MELD' '/kaggle/working/'
!mkdir '/kaggle/working/model'
!cp '/kaggle/input/bert-pretrained/bert_meld_finetune_model.pth' '/kaggle/working/model/bert_meld_finetune_model.pth'


In [6]:
from sklearn.metrics import accuracy_score
from tqdm import tqdm

In [7]:
from torch.utils.data import Dataset
import tensorflow_hub as hub

class MELD_Modal_Dataset(Dataset):
    def __init__(self, audio_dir, text_dir, audio_processor, text_tokenizer):
        """
        初始化多模态数据集。
        :param audio_dir: 音频文件的根目录，例如 '/kaggle/working/MELD/train/audio'
        :param text_dir: 文本文件的根目录，例如 '/kaggle/working/MELD/train/text'
        :param audio_processor: 音频处理函数
        :param text_tokenizer: BERT 或其他文本分词器
        """
        self.audio_dir = audio_dir
        self.text_dir = text_dir
        self.audio_processor = audio_processor
        self.text_tokenizer = text_tokenizer

        # 获取音频文件列表
        self.audio_files = [f for f in os.listdir(audio_dir) if f.endswith(".wav")]

        # 获取文本文件列表
        self.text_files = [f.replace('.wav', '.txt') for f in self.audio_files]

        # 标签映射
        self.label_map = {
            "neutral": 0,
            "joy": 1,
            "sadness": 2,
            "anger": 3,
            "surprise": 4,
            "fear": 5,
            "disgust": 6
        }

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        # 处理音频数据
        audio_file = self.audio_files[idx]
        audio_path = os.path.join(self.audio_dir, audio_file)
        speech_array, sampling_rate = torchaudio.load(audio_path)
        
        # 调整采样率
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
            speech_array = resampler(speech_array)
        
        # 音频处理
        audio_inputs = self.audio_processor(
            speech_array.squeeze(), 
            sampling_rate=16000, 
            return_tensors="pt", 
            padding="max_length", 
            max_length=16000 * 3,
            truncation=True
        )
        audio_input_values = audio_inputs["input_values"].squeeze().numpy()
        audio_mean = audio_input_values.mean()
        audio_std = audio_input_values.std()
        normalized_audio = (audio_input_values - audio_mean) / audio_std

        # 处理文本数据
        text_file = self.text_files[idx]
        text_path = os.path.join(self.text_dir, text_file)
        with open(text_path, 'r') as f:
            text_content = f.read().strip()  # 读取文本内容
        
        # 文本处理
        text_inputs = self.text_tokenizer(
            text_content,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        text_input_ids = text_inputs["input_ids"].squeeze()
        text_attention_mask = text_inputs["attention_mask"].squeeze()

        # 标签
        label = audio_file.split("_")[-1].split(".")[0]
        label_id = self.label_map.get(label, -1)

        return {
            "audio_input_values": normalized_audio,
            "text_input_ids": text_input_ids,
            "text_attention_mask": text_attention_mask,
            "labels": torch.tensor(label_id, dtype=torch.long)
        }

In [8]:
train_audio_dir = "/kaggle/working/MELD/train/audio"
train_text_dir = "/kaggle/working/MELD/train/text"
val_audio_dir = "/kaggle/working/MELD/val/audio"
val_text_dir = "/kaggle/working/MELD/val/text"
test_audio_dir = "/kaggle/working/MELD/test/audio"
test_text_dir = "/kaggle/working/MELD/test/text"

In [9]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h",num_labels=7)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = MELD_Modal_Dataset(train_audio_dir, train_text_dir, processor, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4,  shuffle=True)

val_dataset = MELD_Modal_Dataset(val_audio_dir, val_text_dir, processor, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=4)

test_dataset = MELD_Modal_Dataset(test_audio_dir, test_text_dir, processor, tokenizer)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# load bert 
# 加载 BERT 模型和 Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)


bert_model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(bert_model.config.hidden_size, bert_model.config.num_labels)
)


bert_ckpt_path = "/kaggle/working/model/bert_meld_finetune_model.pth"
bert_model.load_state_dict(torch.load(bert_ckpt_path, map_location=torch.device('cpu')))






model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-10-f8b50f0fa4b7>:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We rec

<All keys matched successfully>

In [ ]:
from safetensors.torch import load_file

config = AutoConfig.from_pretrained("/kaggle/input/msba-group7-ssl-models/Upload_kaggle/Upload_kaggle/model_new/run_2/checkpoint-5440/config.json",
                                    num_labels=7)
model = Wav2Vec2ForPreTraining.from_pretrained("facebook/wav2vec2-base", config=config)
# 加载权重
state_dict = load_file("/kaggle/input/msba-group7-ssl-models/Upload_kaggle/Upload_kaggle/model_new/run_2/checkpoint-5440/model.safetensors")
model.load_state_dict(state_dict)
wav2vec2_model = model
wav2vec2_model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
     torch.nn.Linear(wav2vec2_model.config.hidden_size, wav2vec2_model.config.num_labels)
 )



In [ ]:
from transformers import BertModel, Wav2Vec2Model
class MultimodalClassifier(nn.Module):
    def __init__(self, bert_model, wav2vec2_model):
        super(MultimodalClassifier, self).__init__()
       
        self.bert = bert_model
        
        
        self.wav2vec2 = wav2vec2_model
      
        
        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + self.wav2vec2.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(256, 7)  # 假设是7分类任务
        )
    
    # forward
    def forward(self, text_input, audio_input):
       
        text_outputs = self.bert(**text_input, output_hidden_states=True)
        text_features = text_outputs.hidden_states[-1][:, 0, :]  # 获取[CLS] token的表示
       
        audio_outputs = self.wav2vec2(audio_input, output_hidden_states=True)
        audio_features = audio_outputs.hidden_states[-1][:, 0, :]

        # 将音频和文本特征拼接
        combined_features = torch.cat((text_features, audio_features), dim=-1)

        # 分类
        logits = self.classifier(combined_features)
        return logits

In [ ]:
from transformers import Trainer, TrainingArguments
from dataclasses import dataclass
import numpy as np
os.environ["WANDB_DISABLED"] = "true"  

@dataclass
class MultimodalCollator:
    def __call__(self, batch):
        return {
           
            "text_input_ids": torch.stack([torch.as_tensor(x["text_input_ids"]) for x in batch]),
            "text_attention_mask": torch.stack([torch.as_tensor(x["text_attention_mask"]) for x in batch]),
            "audio_input_values": torch.stack([torch.as_tensor(x["audio_input_values"]) for x in batch]),
            "labels": torch.tensor([x["labels"] for x in batch])
        }


class MultimodalClassifierAdapter(MultimodalClassifier):
    def forward(self, 
               text_input_ids=None, 
               text_attention_mask=None,
               audio_input_values=None,
               labels=None):
        # 将输入适配为原模型需要的格式
        text_input = {
            "input_ids": text_input_ids,
            "attention_mask": text_attention_mask
        }
        logits = super().forward(text_input, audio_input_values)
        
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            
        return {"loss": loss, "logits": logits}


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalClassifier(bert_model, wav2vec2_model).to(device)
optimizer = torch.optim.AdamW([
    {"params": model.bert.parameters(), "lr": 1e-6}, 
    {"params": model.wav2vec2.parameters(), "lr": 1e-6},
    {"params": model.classifier.parameters(), "lr": 1e-5}
], weight_decay=0.1) 


class CustomTrainer(Trainer):
  

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs): 
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss


    def evaluate(self, *args, **kwargs):
        output = super().evaluate(*args, **kwargs)
        self.lr_scheduler.step(output["eval_loss"])  # 根据验证loss更新学习率
        return output


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": (preds == labels).mean()}

# 初始化适配后的模型
model = MultimodalClassifierAdapter(bert_model, wav2vec2_model).to(device)


training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    fp16=True,
    learning_rate=1e-5,  # 实际会被自定义优化器覆盖
    lr_scheduler_type='cosine',
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    logging_steps=100,
    logging_strategy="epoch",
    eval_steps=200,
    load_best_model_at_end = True,
    save_total_limit=4  # 最多保留4个检查点，超出的旧检查点自动删除
)

# 初始化Trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  
    eval_dataset=val_dataset,     
    data_collator=MultimodalCollator(),
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)

from transformers import TrainerCallback

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3):
        self.patience = patience
        self.best_metric = None
        self.wait = 0

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics["eval_accuracy"]
        if self.best_metric is None or current_metric > self.best_metric:
            self.best_metric = current_metric
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                control.should_training_stop = True

trainer.add_callback(EarlyStoppingCallback(patience=3))
# 开始训练
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.846600,1.756456,0.393103
2,1.688300,1.652843,0.393103
3,1.611300,1.607634,0.393103
4,1.570100,1.580471,0.397701
5,1.540200,1.553037,0.416092
6,1.503800,1.527585,0.434483
7,1.479300,1.510462,0.432184
8,1.452600,1.489916,0.450575
9,1.420600,1.474229,0.464368


In [ ]:
test_results = trainer.evaluate(test_dataset)
print(f"Test set evaluation results: {test_results}")

In [ ]:
torch.save(model.state_dict(),"/kaggle/working/multimodal_meld_finetune2.pth")